# Azure Databricks를 사용하여 Lakeflow 작업 구현

> **업계 상황 - NexaTel Communications**
> NexaTel은 500만 명의 모바일 가입자에게 서비스를 제공하는 유럽의 중간 규모 통신 사업자입니다.
> 네트워크 스위칭 인프라는 매시간 **CDR(Call Detail Records)**을 생성합니다.
> 월별 청구가 실행되기 전에 처리, 정리 및 집계되어야 합니다.
> 데이터 엔지니어링 팀은 이 파이프라인을 4개의 **Lakeflow 작업**으로 조정합니다.
> DAG를 구성하는 노트북 작업 (정렬된 순서):
>
> | 작업              | 노트북              |에 따라 다름      | 실행 조건         |
> |-------------------|---------------------|------------------|-------------------|
> | `ingest_cdr`      | `01_ingest_cdr`     | *(진입점)*        | -                 |
> | `cleanse_cdr`     | `02_cleanse_cdr`    | `ingest_cdr`      | 모두 성공         |
> | `aggregate_usage` | `03_aggregate_usage`| `cleanse_cdr`     | 모두 성공         |
> | `error_handler`   | `04_error_handler`  | `cleanse_cdr`     | 하나 이상의 실패  |
>
> 이 노트북의 모든 데모는 이러한 통신 시나리오를 기반으로 합니다.
> 노트북 작업 파일은 `Demo/pipelines/11-lakeflow-jobs/`에 있습니다.

In [0]:
from scripts.setup import setup_11

spark.sql("USE CATALOG trainer_demo")
setup_11(spark)

## 작업 설정 및 구성 생성

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-lakeflow-jobs.create-lakeflow-job.png)

### 데모: NexaTel CDR 파이프라인을 위한 Lakeflow 작업 생성
**트레이너 지침 - Databricks UI에서 작업 만들기:**
1. 사이드바에서 **작업 및 파이프라인** -> **만들기** -> **작업**으로 이동합니다.
2. 작업 이름: `NexaTel CDR Pipeline`.
3. **작업 1 - `ingest_cdr`**
   - 유형: **노트북**
   - 소스: **작업 공간** -> `pipelines/11-lakeflow-jobs/01_ingest_cdr`로 이동
   - 컴퓨팅: **서버리스**
4. **작업 2 - `cleanse_cdr`**
   - 유형: **노트북** -> `02_cleanse_cdr`
   - 다음에 따라 다름: `ingest_cdr` | Run-if: **모두 성공**
5. **작업 3 - `aggregate_usage`**
   - 유형: **노트북** -> `03_aggregate_usage`
   - 다음에 따라 다름: `cleanse_cdr` | Run-if: **모두 성공**
6. **작업 4 - `error_handler`**
   - 유형: **노트북** -> `04_error_handler`
   - 다음에 따라 다름: `cleanse_cdr` | Run-if: **최소 하나 이상 실패**
7. 작업 수준 **매개변수** 추가: `source_catalog=trainer_demo`, `source_schema=demo_11`, `billing_month=2024-12`

**트레이너의 요점 (정렬):**
- DAG 편집기에는 다이아몬드 모양이 표시됩니다. `cleanse_cdr`에는 두 개의 다운스트림 경로가 있습니다.
- 작업 3과 4는 **상호 배타적**입니다. 작업 실행당 하나만 실행됩니다.
- 작업 순서: `ingest_cdr` → `cleanse_cdr` → (`aggregate_usage` 또는 `error_handler`)
- 서버리스 컴퓨팅: 클러스터 준비가 필요 없으며 작업이 즉시 시작됩니다.
- 4개 작업 모두 동일한 작업 수준 매개변수(모든 작업에 전달되는 키-값 쌍)를 공유합니다.
- **작업 소스** 옵션 표시: 버전 제어 노트북을 위한 작업 공간 및 Git 공급자.

In [0]:
# -- Preview raw CDR data: what the job will process --------------------------
# Trainer: show the raw data the job ingests.
# Highlight the intentional quality issues that cleanse_cdr must handle.

from pyspark.sql import functions as F

raw_cdr = spark.read.table("trainer_demo.demo_11.raw_cdr")

print(f"Total CDR records: {raw_cdr.count():,}")
print("\nSchema:")
raw_cdr.printSchema()

print("\nSample records (5 rows):")
raw_cdr.select(
    "cdr_id", "subscriber_id", "call_type",
    "duration_seconds", "termination_code", "roaming"
).show(5, truncate=False)

# Show call type distribution
print("\nCall type distribution:")
raw_cdr.groupBy("call_type").count().orderBy("call_type").show()

# Intentional quality issues
print("=== Quality issues the pipeline must handle ===")
print(f"  NULL subscriber_id    : {raw_cdr.filter(F.col('subscriber_id').isNull()).count():>6,}")
print(f"  Invalid subscriber_id : {raw_cdr.filter(F.col('subscriber_id').isNotNull() & ~F.col('subscriber_id').rlike(r'^SUB-')).count():>6,}")
print(f"  Negative duration     : {raw_cdr.filter(F.col('duration_seconds') < 0).count():>6,}")
print(f"  Zero duration         : {raw_cdr.filter(F.col('duration_seconds') == 0).count():>6,}")


In [0]:
# -- Demonstrate task dependencies and run-if conditions ----------------------
# Trainer: use this to explain WHEN each run-if condition applies.
# Map the six conditions to realistic NexaTel pipeline scenarios.

from pyspark.sql import functions as F

raw_cdr = spark.read.table("trainer_demo.demo_11.raw_cdr")
total        = raw_cdr.count()
null_subs    = raw_cdr.filter(F.col("subscriber_id").isNull()).count()
bad_duration = raw_cdr.filter(F.col("duration_seconds") <= 0).count()
failed_calls = raw_cdr.filter(F.col("termination_code").isin("FAILED","DROPPED")).count()

print("=== NexaTel CDR quality profile ===")
print(f"  Total raw records        : {total:>8,}")
print(f"  NULL subscriber_id       : {null_subs:>8,}  -> cleanse_cdr will reject these")
print(f"  Invalid duration (<= 0)  : {bad_duration:>8,}  -> cleanse_cdr will reject these")
print(f"  Failed/dropped calls     : {failed_calls:>8,}  -> kept in silver; used in KPIs")
print()

# Illustrate the six run-if conditions with telecom examples
conditions = [
    ("All succeeded",          "aggregate_usage after cleanse_cdr -- only bill clean data"),
    ("At least one succeeded", "send partial billing report even if one branch fails"),
    ("None failed",            "send SLA notification regardless of skips"),
    ("All done",               "clean up temp tables after any pipeline outcome"),
    ("At least one failed",    "error_handler fires when cleanse_cdr raises an exception"),
    ("All failed",             "emergency rollback when ALL upstream tasks failed"),
]
print("=== Run-if condition reference ===")
for cond, example in conditions:
    print(f"  {cond:<28} -> {example}")

# Demonstrate dbutils.jobs.taskValues concept
print("""
=== Task values: passing data between tasks ===
In a real Lakeflow Job, Task 1 sets:
  dbutils.jobs.taskValues.set(key="total_raw_records", value=50000)

Task 2 reads it:
  total = dbutils.jobs.taskValues.get(taskKey="ingest_cdr", key="total_raw_records")

This avoids writing coordination data to a table and keeps the job self-contained.
""")


## 작업 트리거 구성

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-lakeflow-jobs.configure-job-triggers.png)

### 데모: NexaTel CDR 파이프라인에 대한 작업 트리거 구성
**시나리오:** NexaTel은 다음을 기반으로 CDR 파이프라인이 자동으로 시작되기를 원합니다.
 Trigger type | NexaTel use case |
---|---|
 **File arrival** | New hourly CDR batch files land in a Unity Catalog volume |
 **Table update** | An upstream network-events job updates `raw_cdr` table |
 **Continuous** | A near-real-time fraud detection job must never pause |

**트레이너 요점 - 파일 도착 트리거:**  
- **작업 및 파이프라인** -> `NexaTel CDR Pipeline` -> **작업 세부정보** -> **트리거 추가** -> **파일 도착**  
- 경로: `/Volumes/trainer_demo/demo_11/cdr_landing/`  
- 새 파일만 실행을 트리거합니다. 동일한 파일 이름을 덮어써도 다시 트리거되지 않습니다.  
- 프로덕션을 위해 외부 위치에서 **파일 이벤트**를 활성화합니다(50개 작업 제한 및 10,000개 파일 한도 제거).  
- 고급: 파일을 일괄적으로 신속하게 삭제하려면 **마지막 변경 후 대기**를 300초로 설정하세요.  
- **정렬:** 파일 도착 트리거는 디렉터리 내 파일을 **lexical(사전식) 순서**로 처리합니다.  
  - 예: 파일 이름이 `CDR_2024-01-01.parquet`, `CDR_2024-01-02.parquet`처럼 날짜 기반이면 자동으로 정렬됩니다.  
  - Auto Loader는 파일을 lexicographically 정렬하지만, 실제 처리 순서는 보장되지 않으므로 파이프라인은 out-of-order 처리를 지원해야 합니다.

**트레이너의 요점 - 테이블 업데이트 트리거:**  
- 트리거 유형: **테이블 업데이트**  
- 모니터 테이블: `trainer_demo.demo_11.raw_cdr`  
- **모든 테이블이 업데이트됨** 또는 **모든 테이블이 업데이트됨**(여러 소스 테이블을 모니터링하는 경우)을 선택합니다.  
- 동적 매개변수 표시: `{{job.trigger.table_update.updated_tables}}`  
- 선택 사항: `{{job.trigger.table_update.trainer_demo.demo_11.raw_cdr.version}}` - 델타 버전을 추적합니다.  
- **정렬:** 테이블 업데이트 트리거는 델타 테이블의 **버전 순서**(commit 순서)에 따라 트리거됩니다.

**트레이너의 요점 - 연속 트리거:**  
- 트리거 유형: **연속**, 작업 재시도 모드: **실패 시**  
- 거의 실시간으로 CDR을 처리해야 하는 사기 감지 스트리밍 작업에 사용됩니다.  
- 비용 영향: 작업은 연중무휴 24시간 실행됩니다. - 서버리스 컴퓨팅을 사용하여 비용을 최적화합니다.  
- 반복되는 실패 시 지수 백오프가 자동으로 적용됩니다(다시 시작 섹션에 표시됨).

In [0]:
# -- File arrival trigger: demonstrate the monitored path and landing zone -----
# Trainer: show the CDR landing volume path that would be configured in the
# trigger dialog. Then simulate a file arriving by writing sample data.

from pyspark.sql import functions as F

# Show existing volumes in demo_11 schema
print("Volumes in trainer_demo.demo_11:")
try:
    spark.sql("SHOW VOLUMES IN trainer_demo.demo_11").show(truncate=False)
except Exception as e:
    print(f"  (no volumes found - create one for the demo): {e}")

# Illustrate: a file arrival trigger would watch this path
trigger_path = "/Volumes/trainer_demo/demo_11/cdr_landing/"
print(f"\nFile arrival trigger would monitor: {trigger_path}")

# Show how a new CDR file would look when it lands
print("\nSimulating a new CDR file landing in the volume...")
sample_cdr = (
    spark.read.table("trainer_demo.demo_11.raw_cdr")
    .limit(100)
    .select("cdr_id", "subscriber_id", "call_type", "duration_seconds", "call_start_ts")
)
print(f"Sample file: {sample_cdr.count()} CDR records ready to be written to the landing volume")
sample_cdr.show(3)

print("""
In a production setup:
  1. Network switches write CDR files to /Volumes/trainer_demo/demo_11/cdr_landing/
  2. File arrival trigger detects the new file (best-effort check every ~1 minute)
  3. Trigger fires the NexaTel CDR Pipeline job automatically
  4. ingest_cdr task reads the new file and validates it

Important: only one job run starts per trigger check window.
           Use 'Wait after last change' to batch multiple files into one run.
""")


In [0]:
# -- Table update trigger: show monitored table history and dynamic params -----
# Trainer: show the Delta table history to illustrate what commit events
# the table_update trigger would respond to. Show dynamic parameter syntax.

from pyspark.sql import functions as F

# Show Delta history for raw_cdr table
print("Delta table history for trainer_demo.demo_11.raw_cdr:")
spark.sql("""
    DESCRIBE HISTORY trainer_demo.demo_11.raw_cdr
    LIMIT 5
""").select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

# Show what the trigger would do for each commit
print("""
=== Table update trigger behaviour for raw_cdr ===

Each Delta commit (write, merge, delete) to raw_cdr creates a new version.
The trigger fires after detecting a new version.

Dynamic parameters available in the triggered job run:

  {{job.trigger.table_update.updated_tables}}
      Value: ["trainer_demo.demo_11.raw_cdr"]

  {{job.trigger.table_update.trainer_demo.demo_11.raw_cdr.version}}
      Value: 3  (the Delta version that triggered this run)

  {{job.trigger.table_update.trainer_demo.demo_11.raw_cdr.commit_timestamp.iso_datetime}}
      Value: 2024-12-01T02:00:00Z

Advanced timing options:
  Minimum time between triggers: 3600 seconds (hourly batch constraint)
  Wait after last change:        300 seconds  (batch rapid writes)
""")

# Show current table version and row count for context
current = spark.read.table("trainer_demo.demo_11.raw_cdr")
print(f"Current raw_cdr: {current.count():,} records")
print("Termination code breakdown (shows the variety of events the trigger would process):")
current.groupBy("termination_code").count().orderBy("termination_code").show()


## 작업 예약

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-lakeflow-jobs.schedule-job.png)

### 데모: NexaTel CDR 파이프라인 예약
**시나리오:** NexaTel 청구에는 세 가지 반복 일정이 필요합니다.
- **매일 밤 02:00 UTC** - 전날의 CDR 처리(규정 마감일)
- **매월 1일 오전 3시(UTC)** - 월간 송장 집계 생성
- **평일 05:00 UTC(= 06:00 CET)** - 운영 센터의 일일 KPI 보고서

**트레이너 지침:**
1. `NexaTel CDR Pipeline` 열기 -> **작업 세부정보** -> **트리거 추가** -> **예약됨**
2. **간단한 일정** 시연: **12시간** 간격 설정 - 시작 시간이 스케줄러에 의해 선택된다는 점을 지적합니다.
3. **고급 일정**으로 전환 -> **크론 구문 표시**를 선택합니다.
4. **시간대**: UTC를 설정합니다 (UTC가 유럽 이동통신사에 대해 DST 문제를 방지하는 이유를 설명).

**NexaTel의 Cron 표현식:**
| Schedule                  | Cron expression         | Field order: SS MM HH DOM MON DOW |
|---------------------------|------------------------|------------------------------------|
| Every night at 02:00 UTC  | `0 0 2 * * ?`          | DOM/DOW = `?` when the other is `*`|
| 1st of month at 03:00 UTC | `0 0 3 1 * ?`          | DOM = 1, DOW = ?                   |
| Weekdays at 05:00 UTC     | `0 0 5 ? * MON-FRI`    | DOW = MON-FRI, DOM = ?             |
| Every 4 hours             | `0 0 */4 * * ?`        | `/` increment operator             |
| Last Friday of month 06:30 UTC | `0 30 6 ? * 6L`    | `L` = last occurrence              |

**정렬:**
- 표는 일정, 크론 표현식, 필드 순서로 정렬되어 있습니다.
- 각 크론 표현식은 NexaTel의 반복 일정에 맞게 명확하게 정렬되어 있습니다.
- DST 모호성을 피하기 위해 유럽 고객에게 서비스를 제공하는 작업에는 UTC를 사용합니다.
- 청구 작업에 대한 **최대 동시 실행 = 1** - 실행이 겹치면 고객에게 이중 청구가 발생합니다.
- Databricks에서 적용하는 최소 간격: 구성에 관계없이 10초.
- 일과 요일 중 하나만 지정할 경우에는 `?`이 필요합니다.

In [0]:
# -- Cron expression examples for NexaTel telecom scheduling scenarios --------
# Trainer: run this cell to display cron expressions with explanations.
# Walk through each field and use the NexaTel examples to make it concrete.

from datetime import datetime, timezone

# Reference table of cron fields
print("=== Cron expression field reference ===")
print(f"  {'Field':<15} {'Required':<10} {'Allowed values':<20} {'Special characters'}")
print(f"  {'-'*15} {'-'*10} {'-'*20} {'-'*25}")
fields = [
    ("Seconds",      "Yes", "0-59",       ", - * /"),
    ("Minutes",      "Yes", "0-59",       ", - * /"),
    ("Hours",        "Yes", "0-23",       ", - * /"),
    ("Day of month", "Yes", "1-31",       ", - * ? / L W"),
    ("Month",        "Yes", "1-12 or JAN-DEC", ", - * /"),
    ("Day of week",  "Yes", "1-7 or SUN-SAT",  ", - * ? / L #"),
    ("Year",         "No",  "1970-2099",  ", - * /"),
]
for name, req, vals, chars in fields:
    print(f"  {name:<15} {req:<10} {vals:<20} {chars}")

# NexaTel schedule examples with explanations
print("\n=== NexaTel CDR Pipeline - schedule examples ===")
schedules = [
    ("0 0 2 * * ?",        "Nightly CDR processing",           "Every day at 02:00 UTC"),
    ("0 0 3 1 * ?",        "Monthly billing run",              "1st of each month at 03:00 UTC"),
    ("0 0 5 ? * MON-FRI",  "Weekday operations KPI report",    "Mon-Fri at 05:00 UTC (= 06:00 CET)"),
    ("0 0 */4 * * ?",      "4-hourly incremental CDR load",    "Every 4 hours starting at midnight"),
    ("0 30 6 ? * 6L",      "Monthly SLA report",               "Last Friday of the month at 06:30 UTC"),
    ("0 0 8 ? * 6#3",      "Third-Friday fraud review",        "3rd Friday of the month at 08:00 UTC"),
]
print(f"\n  {'Cron expression':<26} {'Label':<40} {'Plain English'}")
print(f"  {'-'*26} {'-'*40} {'-'*45}")
for expr, label, desc in schedules:
    print(f"  {expr:<26} {label:<40} {desc}")

print("""
Special character guide:
  *  = every value       (every hour, every day)
  ?  = no specific value (use in DOM or DOW when the other is specified)
  -  = range             (MON-FRI  = weekdays)
  ,  = list              (MON,WED,FRI = three days)
  /  = increment         (*/4 in hours = every 4 hours)
  L  = last              (6L = last Friday of month)
  #  = nth occurrence    (6#3 = 3rd Friday of month)

TIP: maximum concurrent runs = 1 prevents the nightly billing job from
     overlapping if it runs longer than 24 hours during a data incident.
""")


## 채용공고 알림 구성

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-lakeflow-jobs.configure-job-alerts.png)

### 데모: NexaTel CDR 파이프라인에 대한 작업 알림 구성
**시나리오:** NexaTel의 대기 중인 데이터 엔지니어링 팀은 다음과 같은 경우 즉각적인 알림이 필요합니다.
- 야간 CDR 파이프라인 **실패**(결제 데이터가 불완전할 수 있음 - SLA 영향)
- 실행 소요 시간 **> 45분**(SLA 위반 - 일반 런타임은 ~20분)
- 파이프라인 **시작**(규제 감사 로그 요구 사항)

**트레이너 지침 - 작업 수준 알림 구성:**
1. `NexaTel CDR Pipeline` 열기 -> **작업 세부정보** -> **작업 알림** -> **알림 수정**
2. **이메일** 대상 -> 이벤트 추가: **실패** 및 **기간 경고**(임계값: 45분)
3. 전용 감사 이메일 별칭에 **시작** 이벤트 추가
4. **Slack** 또는 **Microsoft Teams** 대상을 추가합니다(워크스페이스 관리자는 관리 설정 -> 알림에서 이를 생성합니다).

**트레이너 지침 - 작업 수준 알림 구성:**
1. `cleanse_cdr` 작업 편집 -> **알림** -> **추가**
2. 이메일 추가 -> 이벤트: **실패**
3. **마지막 재시도까지 알림 음소거** 활성화(3회 재시도 + 최종 실패에 대해 4회 경고 방지)

**NexaTel의 알림 디자인:**
| Event                       | Destination           | Rationale                                 |
|-----------------------------|----------------------|--------------------------------------------|
| Job start                   | Audit email alias    | Regulatory compliance log                  |
| Job failure                 | PagerDuty (webhook)  | Immediate on-call escalation               |
| Task failure (cleanse_cdr)  | Teams channel        | Dev team visibility into which step failed |
| Duration warning > 45 min   | Engineering email    | Capacity planning investigation            |
| Streaming backlog           | PagerDuty            | Real-time fraud job health monitor         |

**정렬:**  
- 표는 이벤트, 알림 대상, 알림 이유 순으로 정렬되어 있습니다.
- 각 알림은 NexaTel의 운영 요구에 맞게 명확하게 정렬되어 있습니다.
- 작업 알림은 SLA, 규제, 실시간 모니터링 등 다양한 목적에 따라 분리되어 있습니다.
- "마지막 재시도까지 알림 음소거" 기능으로 경고 피로를 방지합니다.

**화두 - 피로 경고:**  
3번의 재시도를 통해 마지막 재시도까지 작업 알림을 음소거합니다.  
이 설정 없이 단일 임시 네트워크 블립은 4개의 경고를 생성합니다.  
P50이 아닌 과거 P95 런타임을 기반으로 기간 경고를 사용합니다. 임계값을 P95 + 50%로 설정합니다.

In [0]:
# -- Webhook payload example for NexaTel job failure notification -------------
# Trainer: show what JSON payload Databricks sends to an HTTP webhook.
# NexaTel's internal incident system receives this and creates a ticket.

import json

# Simulated webhook payload sent by Databricks on job failure
sample_payload = {
    "event_type": "jobs.on_failure",
    "workspace_id": "1234567890123456",
    "run": {
        "job_id": 42,
        "run_id": 98765,
        "run_name": "NexaTel CDR Pipeline",
        "run_page_url": "https://adb-1234567890.azuredatabricks.net/jobs/42/runs/98765",
        "state": {
            "life_cycle_state": "TERMINATED",
            "result_state": "FAILED",
            "state_message": "Task cleanse_cdr failed: NullPointerException at line 47"
        },
        "start_time": 1714636800000,
        "end_time":   1714639920000,
        "trigger": "PERIODIC"
    }
}

print("=== Sample webhook payload (jobs.on_failure) ===")
print(json.dumps(sample_payload, indent=2))

# Compute duration from payload timestamps
duration_s = (sample_payload["run"]["end_time"] - sample_payload["run"]["start_time"]) / 1000
print(f"\nComputed run duration: {duration_s:.0f} seconds ({duration_s/60:.1f} minutes)")

print("""
Key fields for NexaTel's incident management system:
  event_type     -> route to correct alert queue (failure vs warning vs backlog)
  run_page_url   -> one-click link to the job run for the on-call engineer
  state_message  -> root cause text for the PagerDuty incident description
  start_time /
  end_time       -> compute duration to detect SLA breach automatically

How to use it:
  1. In Admin Settings -> Notifications -> add webhook destination with your endpoint URL
  2. The endpoint receives POST requests with this JSON body on each event
  3. Parse event_type to route: on_failure -> PagerDuty, on_start -> audit log
  4. Use Azure Logic Apps or Power Automate to convert webhooks into ITSM tickets

Tip: configure up to 3 system destinations per event type per job.
""")


## 자동 다시 시작 구성

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-lakeflow-jobs.configure-job-automatic-restarts.png)

### 데모: NexaTel CDR 파이프라인에 대한 자동 재시작 구성
**시나리오:**  
NexaTel의 CDR 파이프라인은 매일 수백만 개의 레코드를 처리합니다.  
통신 환경에서 흔히 발생하는 일시적인 오류:  
- 피크 시간 동안 Azure 데이터 레이크에서 읽기 네트워크 시간 초과  
- 두 작업이 동시에 쓸 때 일시적인 델타 테이블 잠금 경합  
- 서버리스 풀 용량 제약으로 인해 콜드 스타트 ​​지연이 발생함  

**트레이너 지침 - 작업 수준 재시도 정책:**  
1. `cleanse_cdr` 작업 편집 -> **재시도** 옆에 **+ 추가**  
2. 최대 재시도 횟수: **3** | 재시도 간격: **60000ms**(1분)  
3. 설명: 간격 = 실패한 실행 시작부터 재시도 시작까지의 ms(실패 지점부터가 아님)  
4. 예: 작업이 30초 동안 실행된 후 실패함 -> 실패 후 30초 후에 재시도가 시작됨(60초 - 30초 경과)  

**트레이너 지침 - 시간 초과 기준:**  
1. `aggregate_usage` 작업 편집 -> **메트릭 임계값**  
2. 지표: **실행 기간** | 경고: **30분** | 제한 시간: **60분**  
3. 시간 초과로 인해 작업이 종료되고 재시도 시도 1회로 계산됩니다.  
4. 패턴: 일반 런타임 20분 -> 30분에 경고 -> 60분에 타임아웃  

**트레이너 지침 - 지속적인 작업 재시도(사기 감지 시나리오):**  
1. 새 작업 생성 -> **트리거 추가** -> **연속**, 작업 재시도 모드: **실패 시**  
2. 지수 백오프 알고리즘 표시(아래 코드 셀 참조)  
3. **작업 세부정보** 패널을 통해 연속 작업 모니터링: 연속 실패 + 다음 재시도 시간 표시  

**NexaTel의 생산 모범 사례:**  
| Task              | Retries | Interval | Rationale                        |
|-------------------|---------|----------|-----------------------------------|
| `ingest_cdr`      | 3       | 30 s     | Transient file-read timeout       |
| `cleanse_cdr`     | 3       | 60 s     | Lock contention resolves quickly  |
| `aggregate_usage` | 2       | 120 s    | Heavy compute - allow recovery time |
| `error_handler`   | 0       | -        | Must not retry: append-only logging |

**화두 - 멱등성:**  
- `cleanse_cdr`은 덮어쓰기 모드 -> **멱등성**(여러 번 재시도해도 안전함)을 사용합니다.  
- `error_handler`은 APPEND 모드를 사용합니다. -> **멱등성이 아님**(재시도하면 중복 로그 항목이 생성됨)  
- 비멱등성 작업(송장 전송, 외부 API 호출)은 서버리스에서 자동 최적화를 비활성화해야 합니다.

In [0]:
# -- Simulate task retry behaviour for the cleanse_cdr step ------------------
# Trainer: this cell SIMULATES the retry logic configured on cleanse_cdr.
# Walk through the retry configuration dialog in the UI while this runs.
# In the real Lakeflow Job, the retry is managed by the platform automatically.

import random
import time

def run_cleanse_cdr(attempt: int) -> dict:
    """Simulate cleanse_cdr with decreasing failure probability per retry."""
    # Simulate transient failure: 70% chance on attempt 1, decreasing each time
    fail_probability = max(0.0, 0.70 - (attempt - 1) * 0.30)
    if random.random() < fail_probability:
        raise RuntimeError(
            f"Attempt {attempt}: transient Azure storage timeout reading raw_cdr (simulated)"
        )
    raw_cdr = spark.read.table("trainer_demo.demo_11.raw_cdr")
    rejected = int(raw_cdr.count() * 0.03)   # ~3% rejection rate
    return {
        "attempt":        attempt,
        "raw_records":    raw_cdr.count(),
        "silver_records": raw_cdr.count() - rejected,
        "rejected":       rejected,
    }

MAX_RETRIES      = 3
RETRY_INTERVAL_S = 3   # shortened for demo (real config: 60 000 ms)

print(f"cleanse_cdr task starting (max_retries={MAX_RETRIES}, interval={RETRY_INTERVAL_S}s demo / 60s real)")
print(f"{'='*70}\n")

random.seed(42)   # reproducible output for the demo
result = None
for attempt in range(1, MAX_RETRIES + 2):
    print(f"  Attempt {attempt}/{MAX_RETRIES + 1}: ", end="", flush=True)
    try:
        result = run_cleanse_cdr(attempt)
        print(f"SUCCESS  ->  {result['silver_records']:,} silver records written")
        print(f"\n  Task completed on attempt {attempt} of {MAX_RETRIES + 1}.")
        break
    except RuntimeError as e:
        if attempt <= MAX_RETRIES:
            print(f"FAILED  ->  {e}")
            print(f"            Databricks waits {RETRY_INTERVAL_S}s before retry ...")
            time.sleep(RETRY_INTERVAL_S)
        else:
            print(f"FAILED  ->  All {MAX_RETRIES} retries exhausted.")
            print(f"            Task status set to FAILED. error_handler task will run next.")

if result:
    print(f"\n  Summary: {result['raw_records']:,} raw -> {result['silver_records']:,} silver ({result['rejected']:,} rejected)")


In [0]:
# -- Exponential backoff for continuous jobs (fraud detection scenario) -------
# Trainer: visualise the exponential backoff that Databricks applies when a
# continuous job (e.g. NexaTel real-time fraud detection) fails repeatedly.
# This illustrates the algorithm described in the learn unit.

print("=== Exponential Backoff - NexaTel Fraud Detection (continuous job) ===\n")
print("Scenario: the real-time CDR fraud detection job fails 8 times in a row")
print("due to a degraded network path to the CDR source table.\n")

# Simulate Databricks exponential backoff (illustrative approximation)
BASE_DELAY_S  = 10     # starting delay
MAX_DELAY_S   = 3600   # 1-hour cap (system maximum)

print(f"  {'Run':<5} {'Result':<10} {'Consec Fails':<14} {'Wait before next run (s)':<26} {'Wait (min)'}")
print(f"  {'-'*5} {'-'*10} {'-'*14} {'-'*26} {'-'*10}")

consecutive_failures = 0
for run in range(1, 18):
    # Simulate: runs 1-8 fail, 9+ succeed
    is_failure = run <= 8

    if is_failure:
        consecutive_failures += 1
        wait_s = min(BASE_DELAY_S * (2 ** (consecutive_failures - 1)), MAX_DELAY_S)
        status = "FAILED"
        note = ""
    else:
        consecutive_failures = 0
        wait_s = BASE_DELAY_S  # resets after successful run
        status = "SUCCESS"
        note = " <- backoff resets" if run == 9 else ""

    print(f"  {run:<5} {status:<10} {consecutive_failures:<14} {wait_s:<26} {wait_s/60:<10.1f}{note}")

total_downtime = sum(
    min(BASE_DELAY_S * (2 ** i), MAX_DELAY_S)
    for i in range(8)
)
print(f"\n  Total time in backoff (runs 1-8): {total_downtime}s = {total_downtime/60:.1f} minutes")

print("""
Key points for the audience:
  1. Continuous jobs restart automatically - no manual intervention needed.
  2. Exponential backoff prevents hammering a degraded resource with rapid retries.
  3. After a successful run, the backoff sequence RESETS (back to BASE_DELAY_S).
  4. The 'Job details' panel shows: consecutive failures, time to next retry.
  5. Combine with a Duration warning notification to alert when downtime exceeds SLA.
  6. For non-continuous jobs: configure task-level retry policies (separate UI setting).
""")
